# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [1]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

FileNotFoundError: [Errno 2] No such file or directory: 'train.bin'

## Split dataset into training and test

In [ ]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

# Streaming batching as there is too much data

In [6]:
import numpy as np
from tokenizers import Tokenizer
import random

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

file_path = "train.txt"

def sample_story():
    try:
        with open(file_path, "rb") as f:  # binary mode

            f.seek(0, 2)
            file_size = f.tell()

            pos = random.randint(0, max(1, file_size - 20000))
            f.seek(pos)

            chunk = f.read(20000)

        # decode safely
        text = chunk.decode("utf-8", errors="ignore")

        parts = text.split("endoftext")

        if len(parts) < 2:
            return None

        return random.choice(parts).strip()

    except Exception:
        return None

In [ ]:
def get_batch(block_size, batch_size, stride=None):
    if stride is None:
        stride = block_size // 2

    x_batch = []
    y_batch = []

    while len(x_batch) < batch_size:
        story = sample_story()
        if not story:
            continue

        tokens = tokenizer.encode(story).ids
        if len(tokens) <= block_size + 1:
            continue

        # Build overlapping windows from this story
        for start in range(0, len(tokens) - block_size, stride):
            if len(x_batch) >= batch_size:
                break

            x = tokens[start : start + block_size]
            y = tokens[start + 1 : start + block_size + 1]
            x_batch.append(x)
            y_batch.append(y)

        # In case story length matches exactly block_size, add one window
        if len(tokens) >= block_size + 1 and len(x_batch) < batch_size:
            start = max(0, len(tokens) - block_size - 1)
            x = tokens[start : start + block_size]
            y = tokens[start + 1 : start + block_size + 1]
            x_batch.append(x)
            y_batch.append(y)

    return np.array(x_batch[:batch_size]), np.array(y_batch[:batch_size])

## Traing loop

In [8]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        
        logits, _ = model.forward(idx_cond, np.array([1]))
        
        logits = logits[:, -1, :]
        
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        next_tokens = []
        for b in range(probs.shape[0]):
            next_token = np.random.choice(probs.shape[-1], p=probs[b])
            next_tokens.append(next_token)
        
        next_token = np.array(next_tokens).reshape(-1, 1)
        
        idx = np.concatenate([idx, next_token], axis=1)
    
    return idx

In [9]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model):
    prompt = "History "

    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = generate(model, context, 30)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 256
vocabulary_size = 32_000
block_size = 64
batch_size = 32
block_layers = 4
gradient = Adam(lr=3e-3, warmup_steps=1000, min_lr=1e-4)

model = MiniGPT(vocabulary_size, d_model, block_size, block_layers, gradient)
# model = MiniGPT.__new__(MiniGPT)
# model = model.load("saved_model")
ema_loss = None

for step in range(10_000, 15_001):
    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step % 100 == 0:
        check_model_output(model)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")


Huge story . In this fairy ' s magic jokes appeared . The wand appeared princess started her new words , and the king became very real ! One day , something unexpected happened . A big wind came and blew the rock away . The princess and her friends were surprised . They looked at each other and made a funny face . Then , the girl sprayed her juice started to talk to them . She thought it was magic . The king said , " Thank you for being a nice and frog . I will take a rest for you with you !" The princess and her friends were happy to help . They knew that being kind was the best friends ever with their magic
step 10000, lr 0.003000, loss 2.2611, ema_loss 2.2611
Huge story . They are brave . " Can I read something for her ?" Lily asked . " Yes , please ," she said . " But I don ' t want to ." " I don ' t mean to mom ," Ben said . " warn me . They don ' t like to sleep ." Lily said . " You can understand you ," Anna said . Ben smiled . He wanted to be happy . He climbed the tree and rea

Custom Decoder

In [25]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

# itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 30)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

NameError: name 'stoi' is not defined

## Hugging Face Decoder

In [14]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model):
    prompt = "Hi"

    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = generate(model, context, 128)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

check_model_output(model)

Hi upon a time , there was a girl named Jill . She liked to stir all the smells . One day , her mom made for her . Jill ' s mom asked her , " Can I do bad things ?" Jill said , " But let ' s try to make the spicy , but we can ' t make an alarm ring lots ." Mum said , " No , let ' s ask later ." Jill didn ' t want to listen . Finally they made a plan . Jill saw a big box with a loud noise . She wanted to open it very well , but first , her mom said , " No , that ' s not allowed !" Mia
